In [ ]:
import torch
import pandas as pd 
import numpy as np
import pickle
from datasets import Dataset, DatasetDict
from typing import Dict, Any, Union
from sklearn.metrics import f1_score, accuracy_score
from optuna import Trial
from active_learning_related.modules.smalltext_pipeline import SmallTextPipeline
from setfit import SetFitModel, Trainer, TrainingArguments
import gc
import optuna
import json

In [ ]:
study_name = "depression_hpo"
storage_name = "sqlite:////../data/optuna_logs/{}.db".format(study_name)

# Save study history

In [ ]:
study = optuna.create_study(study_name=study_name, storage=storage_name, load_if_exists=True)
dep_study_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
dep_study_df.to_csv("depression_study.csv")

# Extract Best Trial

In [ ]:
best_trial = study.best_trial
params = best_trial.params

# Prepare Final Training

In [6]:
def compute_metrics(y_pred, y_true):
    from sklearn.metrics import f1_score
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return {
        "f1": f1_score(y_true, y_pred, pos_label=0),
        "accuracy": accuracy_score(y_true, y_pred)
    }

def objective(metrics):
    print(metrics)
    return metrics['f1']

In [ ]:
depression_pipeline = SmallTextPipeline.load_learning_state("depression_state.pkl")

with open('../../data/small_text_datasets.pkl', 'rb') as f:
    small_text_datasets = pickle.load(f)

train_dataset = small_text_datasets['psyC']['smalltext_train_dset']
test_dataset = small_text_datasets['psyC']['smalltext_test_dset']

# Update train dataset with labels of other pipeline

train_dataset.y[depression_pipeline.history[-1]['indices_labeled']] = depression_pipeline.history[-1]['indices_labels']

indices_initial = depression_pipeline.history[-1]['indices_labeled']

train = Dataset.from_dict({
    'text': train_dataset.x,
    'label': train_dataset.y
})

test = Dataset.from_dict({
    'text': test_dataset.x,
    'label': test_dataset.y
})

train_labeled = train.filter(lambda x: x['label'] != -1)

In [ ]:
def model_init(params: Dict[str, Any]) -> SetFitModel:
    params = params or {}
    model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", **params, use_differentiable_head=True)

    # Move model body to GPU if available
    if torch.cuda.is_available():
        model.model_body.to("cuda")

    return model

In [ ]:
training_args = TrainingArguments(
    body_learning_rate = params.get("body_learning_rate"),
    batch_size = params.get("batch_size"),
    num_iterations = params.get("num_iterations"),
    max_steps = params.get("max_steps"),
    seed=1234,
    use_amp=True
)
# Reinitialize model each trial
model = model_init(params= {})

trainer = Trainer(
    model=model,
    train_dataset=train_labeled,
    eval_dataset=test,
    metric=compute_metrics,
    args = training_args
)
trainer.train()

# Evaluate
metrics = trainer.evaluate()

In [ ]:
trainer.model.save_pretrained(save_directory="sentence-transformers/depression_best_model")

config_path = "sentence-transformers/depression_best_model/config.json"

# Load existing config
with open(config_path, "r") as f:
    config = json.load(f)

# Add the missing field
config["_name_or_path"] = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Save it back
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

# Final Predictions

In [ ]:
depression_preds = trainer.model.predict(train["text"], as_numpy=True)
depression_preds_proba = trainer.model.predict_proba(train['text'], as_numpy=True)
depression_preds_df = pd.DataFrame(
    {'predicted_class': depression_preds}
)

depression_preds_proba_df = pd.DataFrame(
    depression_preds_proba
)
depression_preds_df.to_csv("1105_depression_preds.csv")
depression_preds_proba_df.to_csv("1105_depression_preds_proba.csv")